# Bronze Layer - Retail Data Engineering Project

## Purpose
This notebook reads raw retail data from Azure SQL Database and stores it in the Bronze layer.

## Source Tables
- customers
- products
- stores
- transactions

## Bronze Layer Principle
Data is stored in its raw form with no business transformations.

In [0]:
 jdbc_url = (
    "jdbc:sqlserver://sql-retail-sneh.database.windows.net:1433;"
    "database=sqldb-retail;"
    "encrypt=true;"
    "trustServerCertificate=false;"
)

sql_user = "sqladmin"
sql_password = dbutils.secrets.get(
    scope="retail-secrets",
    key="sql-password"
)



In [0]:
display(
    dbutils.fs.ls(
        "abfss://retail@saretailsneh.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://retail@saretailsneh.dfs.core.windows.net/bronze/,bronze/,0,1783195502000
abfss://retail@saretailsneh.dfs.core.windows.net/gold/,gold/,0,1783195537000
abfss://retail@saretailsneh.dfs.core.windows.net/silver/,silver/,0,1783195517000
abfss://retail@saretailsneh.dfs.core.windows.net/source/,source/,0,1783274613000


In [0]:
source_tables = [
    "customers",
    "products",
    "stores",
    "transactions"
]

bronze_base_path = (
    "abfss://retail@saretailsneh.dfs.core.windows.net/bronze"
)

for table in source_tables:

    print(f"Processing {table}...")

    df = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", f"dbo.{table}")
        .option("user", sql_user)
        .option("password", sql_password)
        .option(
            "driver",
            "com.microsoft.sqlserver.jdbc.SQLServerDriver"
        )
        .load()
    )

    row_count = df.count()

    (
        df.write
        .mode("overwrite")
        .parquet(f"{bronze_base_path}/{table}/")
    )

    print(f"✓ {table}: {row_count} rows written to Bronze")

Processing customers...
✓ customers: 5000 rows written to Bronze
Processing products...
✓ products: 500 rows written to Bronze
Processing stores...
✓ stores: 50 rows written to Bronze
Processing transactions...
✓ transactions: 100005 rows written to Bronze


In [0]:
bronze_transactions = spark.read.parquet(
    "abfss://retail@saretailsneh.dfs.core.windows.net/bronze/transactions/"
)
print(f"Bronze transactions count: {bronze_transactions.count():,}")

Bronze transactions count: 100,006
